In [1]:
import os
import math
import torch
from torch.nn.functional import softplus, pad
from datetime import datetime
from tensorboardX import SummaryWriter
from nn.neural_network_dens2 import NeuralNetwork
from training.parse_command_line_arguments import parse_command_line_arguments
from training.util import generate_id, empty_error_dict, compute_error_dict
from training.density_dataset import AtomsDensityData
from training.hamiltonian_dataset import seeded_random_split
from training.exponential_moving_average import ExponentialMovingAverage
from training.lookahead import Lookahead
from training.batch_loader import BatchLoader
from nn.modules.spherical_harmonics_expansion import SphericalHarmonicsExpansion
import numpy as np
import time
from functools import partial
from training.grids import cubical_grid, cubical_sampling
from gradient_learning import utils as grad_utils

%load_ext autoreload
%autoreload 2

In [2]:
directory = '2020-08-20_LuCuJBCl'  # load directory name
model_name = 'LuCuJBCl'

#directory = '2020-07-30_ihEqX0KV'  # load directory name
#model_name = 'ihEqX0KV'

#directory = '2020-04-30_To0wdEze'
checkpoint_dir = os.path.join(
    directory, 'checkpoints')  # checkpoint directory
# load latest checkpoint
checkpoint = torch.load(os.path.join(
    checkpoint_dir, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['epoch']
ID = checkpoint['ID']  # load ID
args = checkpoint['args']  # overwrite args
args.use_gpu = True
args.load_from = os.path.join(directory, 'best_' + model_name + '.pth')

In [3]:
use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + args.dens_dataset + "...")
print("loading atoms from" + args.np_dataset + "...")

# density_file = '/home/mihail/data/water_rot/full_densities.hdf5'
# np_file = 'h2o_overlap_static.npy'

args.num_workers = 0

#args.dens_dataset = 'datasets/h2o_static_pyscf_dft.npy'
#args.np_dataset = 'datasets/h2o_overlap_static_centered.npy'
#args.dens_dataset = 'datasets/h2o_static_pyscf_dft.npy'

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=args.density_subsamples,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype)


equiv_model = NeuralNetwork(load_from=args.load_from)
equiv_model.to(args.dtype)
if args.use_gpu:
    equiv_model.cuda()
len(dataset)

loading density fromdatasets/h2o_dynamic_pyscf_dft.npy...
loading atoms fromdatasets/h2o_dynamic_centered.npy...
Starting atomsdata density init
Some variables
level 2
finished init
saved state dict_keys(['state_dict', 'orbitals', 'order', 'num_features', 'num_basis_functions', 'num_radial_components', 'num_modules', 'num_residual_pre_x', 'num_residual_post_x', 'num_residual_pre_vi', 'num_residual_pre_vj', 'num_residual_post_v', 'num_residual_output', 'basis_functions', 'cutoff', 'activation', 'Zmax'])
An orbital with L=5 was found, but the neural network was initialized with L=6
The neural network SHOULD have at least twice the order of the maximum order orbital for good results!
orbital_spec [[(8, 11, 0), (8, 8, 1), (8, 6, 2), (8, 4, 3), (8, 3, 4), (8, 2, 5)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)]]
cg_matrix shape torch.Size([121, 121, 121])
L_counts [16, 12, 10, 7, 5, 2, 0, 0, 0, 0, 0, 0, 0]
L_dict {(8, 0): r

4999

In [4]:
import time
start = time.time()
sample = dataset[list(range(3))]
#sample = dataset[0]


if use_gpu:
    for key in sample.keys():
        if isinstance(sample[key], torch.Tensor):
            sample[key] = sample[key].cuda()



print(sample.keys())
print(args.dtype)
print(sample['positions'].type())
coeffs = equiv_model(R=sample['positions'])
print(coeffs.keys())
batch_size = sample['positions'].shape[0]
print('elapsed', time.time() - start)

dict_keys(['density', 'coords', 'coord_weights', 'atom_numbers', 'idx', 'positions', '_idx'])
torch.float32
torch.cuda.FloatTensor
dict_keys(['spherical_coeffs', 'radial_width', 'radial_scale'])
elapsed 0.9052159786224365


In [5]:
print('sph coeffs', coeffs['spherical_coeffs'])

sph coeffs [{(8, 0): tensor([[[[ 4.7888e+00,  4.7406e+00,  2.8826e-01,  5.9463e+00,  9.0963e-01,
            1.4763e-01, -9.9874e-01, -1.0368e-02, -2.0137e-01,  3.1178e-03,
            1.2794e+00]]],


        [[[ 4.7889e+00,  4.7407e+00,  2.9219e-01,  5.9457e+00,  9.1087e-01,
            1.4679e-01, -1.0008e+00, -7.3420e-03, -1.9961e-01,  2.2029e-03,
            1.2796e+00]]],


        [[[ 4.7883e+00,  4.7408e+00,  2.9271e-01,  5.9456e+00,  9.1176e-01,
            1.4656e-01, -1.0016e+00, -6.7798e-03, -1.9901e-01,  2.0103e-03,
            1.2799e+00]]]], device='cuda:0', grad_fn=<IndexBackward>), (8, 1): tensor([[[[ 3.4253e-04, -7.3566e-04,  7.7650e-04,  6.9149e-04,  1.1429e-03,
            5.5770e-04,  7.2202e-05,  8.3347e-05],
          [-5.3859e-04,  1.6198e-03, -1.6780e-03, -1.1108e-03, -1.9991e-03,
           -7.6917e-04,  7.0250e-05, -1.0695e-04],
          [-1.5875e-04,  7.2500e-04, -7.3862e-04, -3.4348e-04, -7.0063e-04,
           -1.7472e-04,  1.1910e-04, -1.7709e-05]]],


 

In [6]:
start = time.time()
sph_coeffs = coeffs['spherical_coeffs']
rad_coeffs = coeffs['radial_width']
rad_scale = coeffs['radial_scale']

max_num_coeffs = [0] * (equiv_model.order_max + 1)
max_num_radial = [0] * (equiv_model.order_max + 1)
print('max order', equiv_model.order_max)
for i in range(len(sph_coeffs)):
    for key in sph_coeffs[i].keys():
        L = key[1]
        print(i, L)
        #print(sph_coeffs[i][key])
        if sph_coeffs[i][key].shape[-1] > max_num_coeffs[L]:
            max_num_coeffs[L] = sph_coeffs[i][key].shape[-1]
        if rad_coeffs[i][key].shape[-2] > max_num_radial[L]:
            max_num_radial[L] = rad_coeffs[i][key].shape[-2]
print('max num coeffs', max_num_coeffs)
print('max num radial', max_num_radial)

all_sph = [[torch.zeros([batch_size, 1, 1, max_num_coeffs[i]]).to(sample['positions']) for j in range(len(sph_coeffs))] for i in range(equiv_model.order_max + 1)] 
all_width = [[torch.zeros([batch_size, 1, max_num_radial[i], max_num_coeffs[i]]).to(sample['positions'])  for j in range(len(sph_coeffs))] for i in range(equiv_model.order_max + 1)]
all_scale = [[torch.zeros([batch_size, 1, max_num_radial[i], max_num_coeffs[i]]).to(sample['positions']) for j in range(len(sph_coeffs))] for i in range(equiv_model.order_max + 1)]
print('all width shapes', [[d.shape for d in sca] for sca in all_scale])

for i in range(len(sph_coeffs)):
    for key in sph_coeffs[i].keys():
        L = key[1]
        print('i, ', i, ', key', key)
        sph_norm = sph_coeffs[i][key].norm(dim=-2, keepdim=True)
        
        print('shp norm shape', sph_norm.shape)
        print('max num coeffs', max_num_coeffs[L])
        padding = (0, max_num_coeffs[L] - sph_norm.shape[-1])
        print('padding', padding)
        sph_norm = pad(sph_norm, padding)
        width_pad = pad(rad_coeffs[i][key], padding)
        scale_pad = pad(rad_scale[i][key], padding)
        
        all_sph[L][i] = sph_norm
        all_width[L][i] = width_pad
        all_scale[L][i] = scale_pad
        print('sph_coeffs', sph_norm.shape)
        print('rad_coeffs', width_pad.shape)
        print('rad_scale',  scale_pad.shape)

        print('sph_coeffs', all_sph[L][i].shape)
        print('rad_coeffs', all_width[L][i].shape)
        print('rad_scale', all_scale[L][i].shape)
        print([[d.shape for d in all_sph[l] if d is not None] for l in range(equiv_model.order_max + 1)])
        
        
for i in range(equiv_model.order_max + 1):
    print("L", i)
    
    all_sph[i] = torch.stack(all_sph[i], dim=-1).sum(dim=-1)
    all_width[i] = torch.stack(all_width[i], dim=-1).sum(dim=-1)
    all_scale[i] = torch.stack(all_scale[i], dim=-1).sum(dim=-1)
    print(all_sph[i].shape)
    print(all_width[i].shape)
    print(all_scale[i].shape)

all_sph = torch.cat(all_sph, dim=-1)
all_width = torch.cat(all_width, dim=-1)
all_scale = torch.cat(all_scale, dim=-1)

print(all_sph.shape)
print(all_width.shape)
print(all_scale.shape)
invariant_feats = torch.cat([all_sph, all_width, all_scale], dim=-1)
print(invariant_feats.shape)
print('elapsed', time.time() - start)

max order 5
0 0
0 1
0 2
0 3
0 4
0 5
1 0
1 1
1 2
1 3
1 4
2 0
2 1
2 2
2 3
2 4
max num coeffs [11, 8, 6, 4, 3, 2]
max num radial [1, 1, 1, 1, 1, 1]
all width shapes [[torch.Size([3, 1, 1, 11]), torch.Size([3, 1, 1, 11]), torch.Size([3, 1, 1, 11])], [torch.Size([3, 1, 1, 8]), torch.Size([3, 1, 1, 8]), torch.Size([3, 1, 1, 8])], [torch.Size([3, 1, 1, 6]), torch.Size([3, 1, 1, 6]), torch.Size([3, 1, 1, 6])], [torch.Size([3, 1, 1, 4]), torch.Size([3, 1, 1, 4]), torch.Size([3, 1, 1, 4])], [torch.Size([3, 1, 1, 3]), torch.Size([3, 1, 1, 3]), torch.Size([3, 1, 1, 3])], [torch.Size([3, 1, 1, 2]), torch.Size([3, 1, 1, 2]), torch.Size([3, 1, 1, 2])]]
i,  0 , key (8, 0)
shp norm shape torch.Size([3, 1, 1, 11])
max num coeffs 11
padding (0, 0)
sph_coeffs torch.Size([3, 1, 1, 11])
rad_coeffs torch.Size([3, 1, 1, 11])
rad_scale torch.Size([3, 1, 1, 11])
sph_coeffs torch.Size([3, 1, 1, 11])
rad_coeffs torch.Size([3, 1, 1, 11])
rad_scale torch.Size([3, 1, 1, 11])
[[torch.Size([3, 1, 1, 11]), torch.Size([

In [7]:
print('invariant_featurs', invariant_feats[0])

invariant_featurs tensor([[[ 5.6356e+00,  4.8760e+00,  4.9598e-01,  6.0093e+00,  1.6093e+00,
           1.4763e-01,  9.9874e-01,  1.0368e-02,  2.0137e-01,  3.1178e-03,
           1.2794e+00,  1.7105e-02,  1.2080e-02,  8.9725e-03,  5.6509e-03,
           2.4070e-03,  9.6601e-04,  1.5599e-04,  1.3674e-04,  7.5828e-03,
           4.3992e-03,  9.0131e-03,  8.1496e-03,  2.6335e-03,  7.2249e-05,
           1.4243e-03,  6.7608e-03,  6.2921e-03,  6.6081e-05,  4.9802e-03,
           1.6397e-03,  2.9833e-03,  8.9730e-04,  1.1551e-03,  2.0818e+03,
           5.3389e+00,  2.7998e+02,  6.8241e+01,  4.6712e+00,  0.0000e+00,
           4.2915e+01,  1.7612e+01,  2.2212e+01,  1.0913e+01,  4.5895e+00,
           0.0000e+00,  1.7336e+01,  0.0000e+00,  6.8457e+00,  0.0000e+00,
           0.0000e+00,  2.6979e+01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           1.3333e+01,  1.7681e+01,  1.5073e+01,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  8.4695e+00,
       

In [8]:
torch.allclose(invariant_feats[0], invariant_feats[2])

False

In [9]:
print(grad_utils.get_invariant_features(equiv_model, sample['positions'])[0])

tensor([ 5.6356e+00,  4.8760e+00,  4.9598e-01,  6.0093e+00,  1.6093e+00,
         1.4763e-01,  9.9874e-01,  1.0368e-02,  2.0137e-01,  3.1178e-03,
         1.2794e+00,  1.7105e-02,  1.2080e-02,  8.9725e-03,  5.6509e-03,
         2.4070e-03,  9.6601e-04,  1.5599e-04,  1.3674e-04,  7.5828e-03,
         4.3992e-03,  9.0131e-03,  8.1496e-03,  2.6335e-03,  7.2249e-05,
         1.4243e-03,  6.7608e-03,  6.2921e-03,  6.6081e-05,  4.9802e-03,
         1.6397e-03,  2.9833e-03,  8.9730e-04,  1.1551e-03,  2.0818e+03,
         5.3389e+00,  2.7998e+02,  6.8241e+01,  4.6712e+00,  0.0000e+00,
         4.2915e+01,  1.7612e+01,  2.2212e+01,  1.0913e+01,  4.5895e+00,
         0.0000e+00,  1.7336e+01,  0.0000e+00,  6.8457e+00,  0.0000e+00,
         0.0000e+00,  2.6979e+01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         1.3333e+01,  1.7681e+01,  1.5073e+01,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  8.4695e+00,
         3.1984e+00,  0.0000e+00,  1.7279e+00,  2.2

In [10]:
grad_utils.from_r(equiv_model, sample['positions'])

i 0
i 1
i 2
i 3
i 4
i 5
i 6
i 7
i 8
i 9
i 10
i 11
i 12
i 13
i 14
i 15
i 16
i 17
i 18
i 19
i 20
i 21
i 22
i 23
i 24
i 25
i 26
i 27
i 28
i 29
i 30
i 31
i 32
i 33
i 34
i 35
i 36
i 37
i 38
i 39
i 40
i 41
i 42
i 43
i 44
i 45
i 46
i 47
i 48
i 49
i 50
i 51
i 52
i 53
i 54
i 55
i 56
i 57
i 58
i 59
i 60
i 61
i 62
i 63
i 64
i 65
i 66
i 67
i 68
i 69
i 70
i 71
i 72
i 73
i 74
i 75
i 76
i 77
i 78
i 79
i 80
i 81
i 82
i 83
i 84
i 85
i 86
i 87
i 88
i 89
i 90
i 91
i 92
i 93
i 94
i 95
i 96
i 97
i 98
i 99
i 100
i 101
grad descs shape torch.Size([3, 102, 9])
r_desc.shape torch.Size([3, 102])
r_d_desc.shape torch.Size([3, 102, 9])
Analytical gradients duration: 51.05682349205017
r_flat shape torch.Size([3, 9])
18
displacements shape torch.Size([3, 18, 9])
displacements shape torch.Size([54, 3, 3])
d_num_descs torch.Size([27, 102])
d descs shape torch.Size([3, 102, 9])
grad tensor([-0.1613,  0.1467, -0.3433,  0.2544,  0.1011,  0.2084, -0.0931, -0.2478,
         0.1349], device='cuda:0')
num grad tensor([-0.16

(tensor([[ 5.6356e+00,  4.8760e+00,  4.9598e-01,  6.0093e+00,  1.6093e+00,
           1.4763e-01,  9.9874e-01,  1.0368e-02,  2.0137e-01,  3.1178e-03,
           1.2794e+00,  1.7105e-02,  1.2080e-02,  8.9725e-03,  5.6509e-03,
           2.4070e-03,  9.6601e-04,  1.5599e-04,  1.3674e-04,  7.5828e-03,
           4.3992e-03,  9.0131e-03,  8.1496e-03,  2.6335e-03,  7.2249e-05,
           1.4243e-03,  6.7608e-03,  6.2921e-03,  6.6081e-05,  4.9802e-03,
           1.6397e-03,  2.9833e-03,  8.9730e-04,  1.1551e-03,  2.0818e+03,
           5.3389e+00,  2.7998e+02,  6.8241e+01,  4.6712e+00,  0.0000e+00,
           4.2915e+01,  1.7612e+01,  2.2212e+01,  1.0913e+01,  4.5895e+00,
           0.0000e+00,  1.7336e+01,  0.0000e+00,  6.8457e+00,  0.0000e+00,
           0.0000e+00,  2.6979e+01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           1.3333e+01,  1.7681e+01,  1.5073e+01,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  8.4695e+00,
           3.1984e+00,  0

In [5]:
splits = np.round(np.linspace(0, 4999 - 1, 1000)).astype(int)
print('splits', splits)

splits [   0    5   10   15   20   25   30   35   40   45   50   55   60   65
   70   75   80   85   90   95  100  105  110  115  120  125  130  135
  140  145  150  155  160  165  170  175  180  185  190  195  200  205
  210  215  220  225  230  235  240  245  250  255  260  265  270  275
  280  285  290  295  300  305  310  315  320  325  330  335  340  345
  350  355  360  365  370  375  380  385  390  395  400  405  410  415
  420  425  430  435  440  445  450  455  460  465  470  475  480  485
  490  495  500  505  510  515  520  525  530  535  540  545  550  555
  560  565  570  575  580  585  590  595  600  605  610  615  620  625
  630  635  640  645  650  655  660  665  670  675  680  685  690  695
  700  705  710  715  720  725  730  735  740  745  750  755  760  765
  770  775  780  785  790  795  800  805  810  815  820  825  830  836
  841  846  851  856  861  866  871  876  881  886  891  896  901  906
  911  916  921  926  931  936  941  946  951  956  961  966  971  976

In [6]:
data_batches = []

for i in range(1, len(splits)):
    samples = dataset[list(range(splits[i - 1], splits[i]))]
    if use_gpu:
        for key in samples.keys():
            if isinstance(samples[key], torch.Tensor):
                samples[key] = samples[key].cuda()

    desc, a_d_desc, n_d_desc = grad_utils.from_r(equiv_model, samples['positions'])
    data_batches.append((desc, a_d_desc, n_d_desc))
    np.save('gradient_batches.npy', data_batches, allow_pickle=True)

i 0
i 1
i 2
i 3
i 4
i 5
i 6
i 7
i 8
i 9
i 10
i 11
i 12
i 13
i 14
i 15
i 16
i 17
i 18
i 19
i 20
i 21
i 22
i 23
i 24
i 25
i 26
i 27
i 28
i 29
i 30
i 31
i 32
i 33
i 34
i 35
i 36
i 37
i 38
i 39
i 40
i 41
i 42
i 43
i 44
i 45
i 46
i 47
i 48
i 49
i 50
i 51
i 52
i 53
i 54
i 55
i 56
i 57
i 58
i 59
i 60
i 61
i 62
i 63
i 64
i 65
i 66
i 67
i 68
i 69
i 70
i 71
i 72
i 73
i 74
i 75
i 76
i 77
i 78
i 79
i 80
i 81
i 82
i 83
i 84
i 85
i 86
i 87
i 88
i 89
i 90
i 91
i 92
i 93
i 94
i 95
i 96
i 97
i 98
i 99
i 100
i 101
grad descs shape torch.Size([5, 102, 9])
r_desc.shape torch.Size([5, 102])
r_d_desc.shape torch.Size([5, 102, 9])
Analytical gradients duration: 81.32627058029175
r_flat shape torch.Size([5, 9])
18
displacements shape torch.Size([5, 18, 9])
displacements shape torch.Size([90, 3, 3])
d_num_descs torch.Size([45, 102])
d descs shape torch.Size([5, 102, 9])
grad tensor([-0.1613,  0.1467, -0.3433,  0.2544,  0.1011,  0.2084, -0.0931, -0.2478,
         0.1349], device='cuda:0')
num grad tensor([-0.16

RuntimeError: CUDA out of memory. Tried to allocate 416.00 MiB (GPU 0; 1.96 GiB total capacity; 805.52 MiB already allocated; 30.62 MiB free; 1.50 GiB reserved in total by PyTorch)

In [7]:
len(data_batches)

5

In [8]:
# continue computation

data_batches = np.load('gradient_batches.npy', allow_pickle=True)

for i in range(len(data_batches) + 1, len(splits)):
    torch.cuda.empty_cache()
    samples = dataset[list(range(splits[i - 1], splits[i]))]
    if use_gpu:
        for key in samples.keys():
            if isinstance(samples[key], torch.Tensor):
                samples[key] = samples[key].cuda()
    
    desc, a_d_desc, n_d_desc = grad_utils.from_r(equiv_model, samples['positions'])
    data_batches.append((desc, a_d_desc, n_d_desc))
    np.save('gradient_batches.npy', data_batches, allow_pickle=True)


i 0
i 1
i 2
i 3
i 4
i 5
i 6
i 7
i 8
i 9
i 10
i 11
i 12
i 13
i 14
i 15
i 16
i 17
i 18
i 19
i 20
i 21
i 22
i 23
i 24
i 25
i 26
i 27
i 28
i 29
i 30
i 31
i 32
i 33
i 34
i 35
i 36
i 37
i 38
i 39
i 40
i 41
i 42
i 43
i 44
i 45
i 46
i 47
i 48
i 49
i 50
i 51
i 52
i 53
i 54
i 55
i 56
i 57
i 58
i 59
i 60
i 61
i 62
i 63
i 64
i 65
i 66
i 67
i 68
i 69
i 70
i 71
i 72
i 73
i 74
i 75
i 76
i 77
i 78
i 79
i 80
i 81
i 82
i 83
i 84
i 85
i 86
i 87
i 88
i 89
i 90
i 91
i 92
i 93
i 94
i 95
i 96
i 97
i 98
i 99
i 100
i 101
grad descs shape torch.Size([5, 102, 9])
r_desc.shape torch.Size([5, 102])
r_d_desc.shape torch.Size([5, 102, 9])
Analytical gradients duration: 81.10753107070923
r_flat shape torch.Size([5, 9])
18
displacements shape torch.Size([5, 18, 9])
displacements shape torch.Size([90, 3, 3])


RuntimeError: CUDA out of memory. Tried to allocate 402.00 MiB (GPU 0; 1.96 GiB total capacity; 589.19 MiB already allocated; 344.62 MiB free; 1.20 GiB reserved in total by PyTorch)

In [9]:
len(data_batches)

5